# Experiment - PCA versus no PCA

This notebook combines the TENX95 random-spot comparison with its spatial-block follow-up. It asks whether using all 4,608 image features offers a reliable gain over PCA-256.

The recorded random 80/20 spot-split result was PCA-256 PCC 0.881 / NRMSE 0.443 and no PCA PCC 0.888 / NRMSE 0.429. The +0.007 PCC difference is small; nearby spots can occur on both sides of a random split. The spatial-block follow-up holds out coordinate regions and removes a two-spot buffer. It has no saved numeric result, so only a recorded rerun can support a conclusion.


## Part A - Random spot split
Each cell below follows the original sequence: prepare environment, download TENX95, embed and barcode-align, split spots, fit scaling/PCA/ridge, and plot the saved comparison. A GPU is useful for first-time embedding.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
recorded = pd.DataFrame([{'setting':'PCA 256','PCC':0.881,'NRMSE':0.443},
                         {'setting':'No PCA (4608)','PCC':0.888,'NRMSE':0.429}])
display(recorded)
fig, ax = plt.subplots(1,2,figsize=(9,3.5))
for a,m in zip(ax,['PCC','NRMSE']):
    a.bar(recorded.setting, recorded[m], color=['steelblue','darkorange'])
    a.set_title(m + (' (higher is better)' if m=='PCC' else ' (lower is better)'))
    a.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.show()


# PCA vs. no PCA on the TENX95 slide (Colab)

**Question:** how much information does PCA throw away, and would ridge predict genes better if we skipped PCA and used all 4,608 embedding numbers?

**Why this experiment:** on the 250-spot test run, "PCA with 200 PCs" and "no PCA" gave identical results. With only 200 training spots there are only 200 directions in the data, so 200 PCs keep everything. To see a real difference we need many more training spots than PCs, so this notebook uses **every spot of TENX95 that has a patch: 7,592 spots** (the expression table has 11,845 rows, but only 7,592 spots have an H&E patch).

**What this notebook does**
1. Sets up Colab (GPU, Google Drive, Hugging Face login).
2. Downloads the TENX95 patches (`.h5`) and expression (`.h5ad`) from Hugging Face.
3. Embeds every patch with **GenBio-PathFM** (4,608 numbers per spot), saved to Drive in chunks of 1,024 so a disconnect doesn't lose progress.
4. Matches patches to expression by barcode and applies log1p.
5. Random **spot** split, 80/20 (6,073 train / 1,519 test), then picks the top 50 most variable genes from the **training spots only**.
6. Fits StandardScaler on the training spots (every setting below starts from the same scaled data, so the comparison is fair).
7. For each setting (**PCA 50, 100, 256, 500, 1000** and **no PCA**):
   - **Reconstruction R²** (test spots): how much of an unseen spot's embedding the PCs can rebuild (1 = nothing lost)
   - **Ridge** (RidgeCV, λ chosen per gene by leave-one-out on the training spots) → **RMSE, NRMSE, PCC** on the test spots, averaged over the 50 genes
   - Time taken
8. Results table and plot, saved to Drive.

**How to decide**
- If **no PCA** is clearly better than **PCA 256** on PCC and NRMSE, dropping PCA is worth it.
- If PCA catches up with no PCA by 256–500 PCs, PCA loses almost nothing that matters for genes, so we keep it (faster, and matches HEST).

**Caveat:** this is a random spot split on one slide, which leaks (neighbouring spots land in both train and test). Its scores are inflated and are only used to compare settings with each other.

**Where it runs:** Google Colab with a **GPU** (Runtime → Change runtime type → **T4 GPU**).

**Before you run it:** a Hugging Face account that has accepted the terms for `MahmoodLab/hest` and can download `genbio-ai/genbio-pathfm`.

**What it saves** (in `MyDrive/BDD_BJMJ/pca_test/`)

| File | Contents |
|---|---|
| `hest_data/patches/TENX95.h5`, `hest_data/st/TENX95.h5ad` | Downloaded HEST files |
| `embeddings_chunks/chunk_*.npy` | GenBio-PathFM embeddings, 1,024 spots per file |
| `TENX95_pca_vs_nopca.csv` | One row per setting: reconstruction R², variance kept, RMSE, NRMSE, PCC, median λ, time |
| `TENX95_pcc_per_gene_by_setting.csv` | PCC of each of the 50 genes under each setting |
| `TENX95_pca_vs_nopca.png` | The three-panel plot (reconstruction R², PCC, NRMSE vs. number of PCs) |

## 1. Setup

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
%pip install -q anndata h5py "pandas==2.2.3"

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
from os import environ
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import anndata
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Google Drive:** everything slow (downloads, embeddings) is saved here, so you only do it once.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = Path('/content/drive/MyDrive/BDD_BJMJ/pca_test')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", WORK_DIR)

**Hugging Face login:** paste your token into the prompt box only. Never put it in the code or upload it to GitHub.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
from huggingface_hub import login
login()

## Settings

- `PCA_SIZES`: the PCA settings to compare (256 = HEST)
- `ALPHAS`: λ values ridge tries (same as `ridge_on_pca.ipynb`)
- `USE_FP16`: half precision on the GPU, about 2× faster embedding; set to `False` to match our earlier embeddings exactly

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
sample_id = "TENX95"
N_GENES = 50
TEST_FRACTION = 0.2
SEED = 0

PCA_SIZES = [50, 100, 256, 500, 1000]
ALPHAS = np.logspace(-2, 5, 30)
MIN_TEST_SD = 0.05

BATCH_SIZE = 64
CHUNK_SIZE = 1024
USE_FP16 = True

## 2. Download TENX95 from Hugging Face

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
from huggingface_hub import hf_hub_download

data_dir = WORK_DIR / "hest_data"

patches_path = hf_hub_download(
    repo_id="MahmoodLab/hest", repo_type="dataset",
    filename=f"patches/{sample_id}.h5", local_dir=data_dir,
)
st_path = hf_hub_download(
    repo_id="MahmoodLab/hest", repo_type="dataset",
    filename=f"st/{sample_id}.h5ad", local_dir=data_dir,
)
print(patches_path)
print(st_path)

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
with h5py.File(patches_path, "r") as f:
    n_spots = f["img"].shape[0]
    raw_barcodes = f["barcode"][:]
    coords = f["coords"][:]
    print("Patches:", f["img"].shape)

barcodes = []
for b in raw_barcodes:
    if hasattr(b, "ravel"):
        b = b.ravel()[0]
    if isinstance(b, bytes):
        b = b.decode()
    barcodes.append(b)
barcodes = np.array(barcodes)

print("Spots on the slide:", n_spots)
print("First barcodes:", barcodes[:3])

## 3. Embed every patch with GenBio-PathFM

Same model and image transform as our earlier notebooks. Patches are read from the file 1,024 at a time (not all at once, to save memory), and each chunk of embeddings is saved to Drive.
If Colab disconnects, just re-run this cell: finished chunks are skipped.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
emb_dir = WORK_DIR / "embeddings_chunks"
emb_dir.mkdir(parents=True, exist_ok=True)

chunk_starts = list(range(0, n_spots, CHUNK_SIZE))
missing = []
for start in chunk_starts:
    if not Path(emb_dir / f"chunk_{start:06d}.npy").exists():
        missing.append(start)
print("Chunks total:", len(chunk_starts), " still to do:", len(missing))

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
if len(missing) > 0:
    from PIL import Image
    from torchvision import transforms
    from transformers import AutoModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModel.from_pretrained("genbio-ai/genbio-pathfm", trust_remote_code=True)
    model = model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.697, 0.575, 0.728), std=(0.188, 0.240, 0.187)),
    ])

    t0 = time.time()
    with h5py.File(patches_path, "r") as f:
        for start in missing:
            end = min(start + CHUNK_SIZE, n_spots)
            chunk_imgs = f["img"][start:end]

            chunk_embeddings = []
            for b in range(0, len(chunk_imgs), BATCH_SIZE):
                batch = [transform(Image.fromarray(img).convert("RGB")) for img in chunk_imgs[b:b + BATCH_SIZE]]
                x = torch.stack(batch).to(device)
                with torch.inference_mode():
                    if USE_FP16 and device.type == "cuda":
                        with torch.autocast("cuda", dtype=torch.float16):
                            out = model(x)
                    else:
                        out = model(x)
                chunk_embeddings.append(out.float().cpu().numpy())

            chunk_embeddings = np.concatenate(chunk_embeddings, axis=0)
            np.save(emb_dir / f"chunk_{start:06d}.npy", chunk_embeddings)
            print(f"spots {start}-{end} done  ({(time.time() - t0) / 60:.1f} min)", flush=True)

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
embeddings = []
for start in chunk_starts:
    embeddings.append(np.load(emb_dir / f"chunk_{start:06d}.npy"))
embeddings = np.concatenate(embeddings, axis=0)

assert len(embeddings) == n_spots
print("Embeddings:", embeddings.shape)

## 4. Load expression and line it up with the patches

Same as before: reorder the expression rows by barcode, then log1p.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
adata = anndata.read_h5ad(st_path)
print("Expression table:", adata.shape, "(spots x genes)")

keep = np.isin(barcodes, adata.obs_names)
print("Patches with a matching expression row:", keep.sum(), "/", len(barcodes))

X_all = embeddings[keep]
barcodes_kept = barcodes[keep]
coords_kept = coords[keep]

counts = adata[barcodes_kept].X
if hasattr(counts, "toarray"):
    counts = counts.toarray()
log_expr = np.log1p(np.asarray(counts, dtype=np.float32))
all_gene_names = np.array(adata.var_names)

print("X:", X_all.shape)
print("log1p expression:", log_expr.shape)

## 5. Split, then pick the top 50 genes from the training spots only

Random spot split, 80/20. Picking genes from training spots only fixes the small leak in our earlier notebooks.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
spot_index = np.arange(len(X_all))
train_idx, test_idx = train_test_split(spot_index, test_size=TEST_FRACTION, random_state=SEED)

gene_variance = log_expr[train_idx].var(axis=0)
top_gene_idx = np.argsort(gene_variance)[::-1][:N_GENES]
gene_names = all_gene_names[top_gene_idx]

X_train = X_all[train_idx]
X_test = X_all[test_idx]
y_train = log_expr[train_idx][:, top_gene_idx]
y_test = log_expr[test_idx][:, top_gene_idx]

print("Train spots:", len(train_idx))
print("Test spots:", len(test_idx))
print("Top 10 genes:", list(gene_names[:10]))

## 6. StandardScaler (fit on train only)

Every setting below, with or without PCA, starts from these scaled embeddings, so the comparison is fair.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
scaler = StandardScaler()
Z_train = scaler.fit_transform(X_train)
Z_test = scaler.transform(X_test)
print("Scaled train:", Z_train.shape, " scaled test:", Z_test.shape)

## 7. Compare PCA sizes and no PCA

For each setting:
- **Reconstruction R²** (test spots): how much of the scaled embedding the PCs can rebuild. 1 = nothing lost. Not defined for "no PCA" (nothing is compressed).
- **Ridge** with λ picked by leave-one-out cross-validation on training spots, one λ per gene
- **RMSE, NRMSE, PCC** on the test spots, per gene, then averaged (same functions as `ridge_on_pca.ipynb`)
- **Time** for PCA + ridge

The mean predictor (each gene's training average) is the baseline.

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
def rmse_one_gene(true_values, predicted_values):
    errors = true_values - predicted_values
    return np.sqrt(np.mean(errors ** 2))


def pcc_one_gene(true_values, predicted_values):
    if np.std(true_values) < 1e-6 or np.std(predicted_values) < 1e-6:
        return np.nan
    return np.corrcoef(true_values, predicted_values)[0, 1]


def score_all_genes(y_true, y_pred):
    rmses, nrmses, pccs = [], [], []
    for g in range(y_true.shape[1]):
        true_values = y_true[:, g]
        predicted_values = y_pred[:, g]
        rmse = rmse_one_gene(true_values, predicted_values)
        sd = np.std(true_values)
        rmses.append(rmse)
        if sd < MIN_TEST_SD:
            nrmses.append(np.nan)
            pccs.append(np.nan)
        else:
            nrmses.append(rmse / sd)
            pccs.append(pcc_one_gene(true_values, predicted_values))
    return np.nanmean(rmses), np.nanmean(nrmses), np.nanmean(pccs), np.array(pccs)


def reconstruction_r2(Z, Z_rebuilt):
    leftover = np.sum((Z - Z_rebuilt) ** 2)
    total = np.sum((Z - Z.mean(axis=0)) ** 2)
    return 1 - leftover / total

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
results = []
pcc_per_gene = {}

y_pred_mean = np.tile(y_train.mean(axis=0), (len(y_test), 1))
rmse, nrmse, pcc, _ = score_all_genes(y_test, y_pred_mean)
results.append({"setting": "mean predictor", "n_features": 0, "recon_R2_test": np.nan,
                "variance_kept_train": np.nan, "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                "median_alpha": np.nan, "seconds": 0})

for k in PCA_SIZES:
    t0 = time.time()
    pca = PCA(n_components=k, random_state=SEED)
    P_train = pca.fit_transform(Z_train)
    P_test = pca.transform(Z_test)
    r2 = reconstruction_r2(Z_test, pca.inverse_transform(P_test))

    ridge = RidgeCV(alphas=ALPHAS, alpha_per_target=True)
    ridge.fit(P_train, y_train)
    y_pred = ridge.predict(P_test)
    rmse, nrmse, pcc, pccs = score_all_genes(y_test, y_pred)
    pcc_per_gene[f"PCA {k}"] = pccs

    results.append({"setting": f"PCA {k}", "n_features": k, "recon_R2_test": r2,
                    "variance_kept_train": pca.explained_variance_ratio_.sum(),
                    "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                    "median_alpha": float(np.median(ridge.alpha_)), "seconds": time.time() - t0})
    print(f"PCA {k:>4}: recon R2 {r2:.3f} | PCC {pcc:.3f} | NRMSE {nrmse:.3f} | {time.time() - t0:.0f} s", flush=True)

t0 = time.time()
ridge = RidgeCV(alphas=ALPHAS, alpha_per_target=True)
ridge.fit(Z_train, y_train)
y_pred = ridge.predict(Z_test)
rmse, nrmse, pcc, pccs = score_all_genes(y_test, y_pred)
pcc_per_gene["No PCA"] = pccs
results.append({"setting": "No PCA", "n_features": Z_train.shape[1], "recon_R2_test": np.nan,
                "variance_kept_train": 1.0, "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                "median_alpha": float(np.median(ridge.alpha_)), "seconds": time.time() - t0})
print(f"No PCA ({Z_train.shape[1]}): PCC {pcc:.3f} | NRMSE {nrmse:.3f} | {time.time() - t0:.0f} s")

## 8. Results

- **recon_R2_test**: share of an unseen spot's scaled embedding the PCs rebuild
- **variance_kept_train**: share of the training variance the PCs keep (the cumulative scree number)
- **RMSE / NRMSE / PCC**: gene prediction on the test spots, averaged over the 50 genes

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
results_table = pd.DataFrame(results)
print(results_table.round(3).to_string(index=False))

results_table.to_csv(WORK_DIR / f"{sample_id}_pca_vs_nopca.csv", index=False)
pd.DataFrame(pcc_per_gene, index=gene_names).to_csv(WORK_DIR / f"{sample_id}_pcc_per_gene_by_setting.csv")
print("Saved CSVs to", WORK_DIR)

**Workflow overview:** This block advances the random-split experiment through setup, download, embedding, alignment, training-only feature processing, scoring, or plotting.

In [ ]:
pca_rows = results_table[results_table["setting"].str.startswith("PCA")]
no_pca = results_table[results_table["setting"] == "No PCA"].iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(pca_rows["n_features"], pca_rows["recon_R2_test"], marker="o")
axes[0].set_xlabel("Number of PCs")
axes[0].set_ylabel("Test reconstruction R²")
axes[0].set_title("How much of the embedding PCA keeps")
axes[0].set_ylim(0, 1)

axes[1].plot(pca_rows["n_features"], pca_rows["PCC"], marker="o", label="PCA")
axes[1].axhline(no_pca["PCC"], color="red", linestyle="--", label=f"No PCA ({no_pca['PCC']:.3f})")
axes[1].set_xlabel("Number of PCs")
axes[1].set_ylabel("Mean PCC (test)")
axes[1].set_title("Gene prediction: PCC (higher is better)")
axes[1].legend()

axes[2].plot(pca_rows["n_features"], pca_rows["NRMSE"], marker="o", label="PCA")
axes[2].axhline(no_pca["NRMSE"], color="red", linestyle="--", label=f"No PCA ({no_pca['NRMSE']:.3f})")
axes[2].set_xlabel("Number of PCs")
axes[2].set_ylabel("Mean NRMSE (test)")
axes[2].set_title("Gene prediction: NRMSE (lower is better)")
axes[2].legend()

for ax in axes:
    ax.axvline(256, color="grey", linewidth=0.8, linestyle=":")

plt.tight_layout()
plt.savefig(WORK_DIR / f"{sample_id}_pca_vs_nopca.png", dpi=150, bbox_inches="tight")
plt.show()

### How to read it

- If **No PCA** is clearly better than **PCA 256** on PCC and NRMSE, dropping PCA is worth it.
- If the PCA line reaches the red dashed line by 256 or 500 PCs, PCA loses almost nothing that matters for genes, and keeping it is faster and matches HEST.
- Reconstruction R² can stay well below 1 while PCC barely changes: PCA may throw away parts of the embedding that don't relate to these 50 genes.
- This is still a **random spot split** (leaky). If No PCA wins, repeat on the patient split to make sure the gain isn't just better memorizing of neighbouring spots.

## Part B - Spatial-block follow-up
This section clusters tissue coordinates into five regions, holds one region out per fold, buffers nearby spots out of training, and compares PCA settings on the same test region.

# PCA vs. no PCA on TENX95, with a spatial-block split (Colab)

**Question:** is ridge on all 4,608 GenBio-PathFM numbers really better than ridge on PCA 256, once the split stops leaking?

**Why this notebook:** in `pca_vs_nopca_colab.ipynb`, no PCA beat PCA 256 by only 0.007 PCC (0.888 vs 0.881), on a random spot split. A random split puts neighbouring spots in train and test, so a model can score well by memorizing its neighbours. A bigger model (4,608 features) can memorize more, so that leak may favour no PCA.

**What is different from the last notebook**

| | Last notebook | This notebook |
|---|---|---|
| Split | random spots, 80/20 | **5 spatial blocks**: hold out one whole region at a time |
| Buffer | none | training spots within **2 spots** of the test block are dropped |
| Choosing λ | leave-one-out on training spots (also leaks) | **inner block CV**: training spots split into 4 regions, λ picked on held-out regions |
| Repeats | 1 split | 5 folds, compared **in pairs** (same fold, both settings) |
| Embeddings | computed on the GPU | **loaded from Drive** (already computed, no 14-min wait) |

Everything that learns from data (gene choice, scaler, PCA, λ) is fit on the training spots of each fold only.

**Decision rule (set before running):** drop PCA only if no PCA beats PCA 256 by **at least 0.01 mean PCC** AND wins in **at least 4 of 5 folds**. Otherwise keep PCA 256.

**Runtime:** a few minutes. A GPU makes the ridge fits faster but is not required.

**What it saves** (in `<your folder>/spatial_blocks/`)

| File | Contents |
|---|---|
| `TENX95_blocks_per_fold.csv` | One row per fold and setting: PCC, NRMSE, RMSE, spot counts, median λ, time |
| `TENX95_blocks_paired.csv` | One row per fold: no PCA minus PCA 256 (PCC and NRMSE), and genes won |
| `TENX95_blocks.png` | Block map, PCC per fold, and the paired differences |

## 1. Setup

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
!pip install -q anndata h5py "pandas==2.2.3"

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
from os import environ
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import anndata
import torch
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Settings

- `N_BLOCKS`: number of regions the slide is cut into (one is held out per fold)
- `N_INNER_BLOCKS`: regions used inside the training spots to pick λ
- `BUFFER_IN_SPOTS`: training spots this many spots or closer to the test block are dropped
- `PCA_SIZES`: PCA settings to compare with no PCA
- `MIN_PCC_GAIN`, `MIN_FOLD_WINS`: the decision rule

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
sample_id = "TENX95"
N_GENES = 50
SEED = 0

N_BLOCKS = 5
N_INNER_BLOCKS = 4
BUFFER_IN_SPOTS = 2

PCA_SIZES = [256, 500]
ALPHAS = np.logspace(-2, 5, 30)
MIN_TEST_SD = 0.05

MIN_PCC_GAIN = 0.01
MIN_FOLD_WINS = 4

**Find the folder with the saved embeddings.** The last run saved to `MyDrive/BDD_pca_test`. If your files are somewhere else, add that folder to the list.

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
possible_dirs = [
    Path('/content/drive/MyDrive/BDD_pca_test'),
    Path('/content/drive/MyDrive/BDD_BJMJ/pca_test'),
]

DATA_DIR = None
for folder in possible_dirs:
    if (folder / "embeddings_chunks").exists():
        DATA_DIR = folder
        break

assert DATA_DIR is not None, "No embeddings_chunks folder found. Check where the last notebook saved its files."

OUT_DIR = DATA_DIR / "spatial_blocks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reading from:", DATA_DIR)
print("Saving to:", OUT_DIR)

## 2. Load embeddings, barcodes, coordinates and expression

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
patches_path = DATA_DIR / "hest_data" / "patches" / f"{sample_id}.h5"
st_path = DATA_DIR / "hest_data" / "st" / f"{sample_id}.h5ad"

with h5py.File(patches_path, "r") as f:
    raw_barcodes = f["barcode"][:]
    coords = f["coords"][:]

barcodes = []
for b in raw_barcodes:
    if hasattr(b, "ravel"):
        b = b.ravel()[0]
    if isinstance(b, bytes):
        b = b.decode()
    barcodes.append(b)
barcodes = np.array(barcodes)

emb_dir = DATA_DIR / "embeddings_chunks"
chunk_files = sorted(name for name in [item.name for item in emb_dir.iterdir()] if name.startswith("chunk_"))
embeddings = np.concatenate([np.load(emb_dir / (name)) for name in chunk_files], axis=0)

assert len(embeddings) == len(barcodes), "Embeddings and patches don't line up"
print("Chunks:", len(chunk_files), " embeddings:", embeddings.shape)

adata = anndata.read_h5ad(st_path)
keep = np.isin(barcodes, adata.obs_names)

X_all = embeddings[keep]
coords_kept = coords[keep].astype(float)
counts = adata[barcodes[keep]].X
if hasattr(counts, "toarray"):
    counts = counts.toarray()
log_expr = np.log1p(np.asarray(counts, dtype=np.float32))
all_gene_names = np.array(adata.var_names)

print("X:", X_all.shape, " log1p expression:", log_expr.shape)

## 3. Cut the slide into spatial blocks

KMeans on the spot coordinates gives 5 compact regions. Spot spacing is the median distance to the nearest spot; the buffer is `BUFFER_IN_SPOTS` times that.

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
blocks = KMeans(n_clusters=N_BLOCKS, random_state=SEED, n_init=10).fit_predict(coords_kept)

nn = NearestNeighbors(n_neighbors=2).fit(coords_kept)
distances, _ = nn.kneighbors(coords_kept)
spot_spacing = np.median(distances[:, 1])
buffer_distance = BUFFER_IN_SPOTS * spot_spacing * 1.01

print("Spots per block:", np.bincount(blocks))
print(f"Spot spacing: {spot_spacing:.1f} px   buffer: {buffer_distance:.1f} px")

plt.figure(figsize=(5, 5))
plt.scatter(coords_kept[:, 0], coords_kept[:, 1], c=blocks, cmap="tab10", s=3)
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("Spatial blocks (one is the test set per fold)")
plt.show()

## 4. Helper functions

**Scoring** is the same as the last notebook.

**Ridge for many λ at once:** for one training set, ridge predictions for every λ come from one SVD of the training features. That is much faster than refitting 30 times, and it gives exactly the same answer as sklearn's `Ridge` (with intercept).

**Choosing λ:** the training spots are split into `N_INNER_BLOCKS` regions. For each λ, predict each region from the others and add up the squared errors. Each gene gets the λ with the smallest error.

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
def rmse_one_gene(true_values, predicted_values):
    errors = true_values - predicted_values
    return np.sqrt(np.mean(errors ** 2))


def pcc_one_gene(true_values, predicted_values):
    if np.std(true_values) < 1e-6 or np.std(predicted_values) < 1e-6:
        return np.nan
    return np.corrcoef(true_values, predicted_values)[0, 1]


def score_all_genes(y_true, y_pred):
    rmses, nrmses, pccs = [], [], []
    for g in range(y_true.shape[1]):
        true_values = y_true[:, g]
        predicted_values = y_pred[:, g]
        rmse = rmse_one_gene(true_values, predicted_values)
        sd = np.std(true_values)
        rmses.append(rmse)
        if sd < MIN_TEST_SD:
            nrmses.append(np.nan)
            pccs.append(np.nan)
        else:
            nrmses.append(rmse / sd)
            pccs.append(pcc_one_gene(true_values, predicted_values))
    return np.nanmean(rmses), np.nanmean(nrmses), np.nanmean(pccs), np.array(pccs)


def ridge_predictions_all_alphas(X_train, y_train, X_new, alphas):
    # returns shape (number of alphas, number of new spots, number of genes)
    x_mean = X_train.mean(axis=0)
    y_mean = y_train.mean(axis=0)

    X_train_c = torch.tensor(X_train - x_mean, dtype=torch.float32, device=device)
    X_new_c = torch.tensor(X_new - x_mean, dtype=torch.float32, device=device)
    y_train_c = torch.tensor(y_train - y_mean, dtype=torch.float32, device=device)

    U, s, Vt = torch.linalg.svd(X_train_c, full_matrices=False)
    A = X_new_c @ Vt.T
    B = U.T @ y_train_c

    predictions = []
    for alpha in alphas:
        shrink = s / (s ** 2 + alpha)
        pred = (A * shrink) @ B
        predictions.append(pred.cpu().numpy() + y_mean)
    return np.stack(predictions)


def choose_alpha_index_per_gene(X_train, y_train, inner_groups, alphas):
    squared_errors = np.zeros((len(alphas), y_train.shape[1]))
    for group in np.unique(inner_groups):
        is_val = inner_groups == group
        preds = ridge_predictions_all_alphas(X_train[~is_val], y_train[~is_val], X_train[is_val], alphas)
        errors = preds - y_train[is_val]
        squared_errors += (errors ** 2).sum(axis=1)
    return np.argmin(squared_errors, axis=0)


def fit_and_predict(X_train, y_train, X_test, inner_groups):
    best_index = choose_alpha_index_per_gene(X_train, y_train, inner_groups, ALPHAS)
    preds_all = ridge_predictions_all_alphas(X_train, y_train, X_test, ALPHAS)
    y_pred = np.zeros((len(X_test), y_train.shape[1]))
    for g in range(y_train.shape[1]):
        y_pred[:, g] = preds_all[best_index[g], :, g]
    return y_pred, best_index

## 5. Run the 5 folds

For each fold: hold out one block, drop training spots inside the buffer, pick the top 50 genes from the training spots, scale, then run every setting on the **same** spots.

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
fold_rows = []
pcc_per_gene = {}

for test_block in range(N_BLOCKS):
    test_idx = np.where(blocks == test_block)[0]
    other_idx = np.where(blocks != test_block)[0]

    nn_test = NearestNeighbors(n_neighbors=1).fit(coords_kept[test_idx])
    dist_to_test, _ = nn_test.kneighbors(coords_kept[other_idx])
    train_idx = other_idx[dist_to_test[:, 0] > buffer_distance]
    n_dropped = len(other_idx) - len(train_idx)

    gene_variance = log_expr[train_idx].var(axis=0)
    top_gene_idx = np.argsort(gene_variance)[::-1][:N_GENES]
    y_train = log_expr[train_idx][:, top_gene_idx]
    y_test = log_expr[test_idx][:, top_gene_idx]

    scaler = StandardScaler()
    Z_train = scaler.fit_transform(X_all[train_idx])
    Z_test = scaler.transform(X_all[test_idx])

    inner_groups = KMeans(n_clusters=N_INNER_BLOCKS, random_state=SEED, n_init=10).fit_predict(coords_kept[train_idx])

    print(f"Fold {test_block}: train {len(train_idx)} (buffer dropped {n_dropped}), test {len(test_idx)}")

    y_pred_mean = np.tile(y_train.mean(axis=0), (len(y_test), 1))
    rmse, nrmse, pcc, _ = score_all_genes(y_test, y_pred_mean)
    fold_rows.append({"fold": test_block, "setting": "mean predictor", "n_train": len(train_idx),
                      "n_test": len(test_idx), "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                      "median_alpha": np.nan, "alphas_at_grid_edge": 0, "seconds": 0})

    settings = []
    for k in PCA_SIZES:
        settings.append((f"PCA {k}", k))
    settings.append(("No PCA", None))

    for name, k in settings:
        t0 = time.time()
        if k is None:
            F_train, F_test = Z_train, Z_test
        else:
            pca = PCA(n_components=k, random_state=SEED)
            F_train = pca.fit_transform(Z_train)
            F_test = pca.transform(Z_test)

        y_pred, best_index = fit_and_predict(F_train, y_train, F_test, inner_groups)
        rmse, nrmse, pcc, pccs = score_all_genes(y_test, y_pred)
        pcc_per_gene[(test_block, name)] = pccs

        at_edge = np.sum((best_index == 0) | (best_index == len(ALPHAS) - 1))
        seconds = time.time() - t0
        fold_rows.append({"fold": test_block, "setting": name, "n_train": len(train_idx),
                          "n_test": len(test_idx), "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                          "median_alpha": float(np.median(ALPHAS[best_index])),
                          "alphas_at_grid_edge": int(at_edge), "seconds": seconds})
        print(f"   {name:>8}: PCC {pcc:.3f} | NRMSE {nrmse:.3f} | {seconds:.0f} s", flush=True)

fold_table = pd.DataFrame(fold_rows)

## 6. Results

**Per-fold table**, then the **paired comparison**: for each fold, no PCA minus PCA 256. Positive PCC difference and negative NRMSE difference mean no PCA did better.

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
print(fold_table.round(3).to_string(index=False))

print()
print("Mean over folds:")
print(fold_table.groupby("setting")[["PCC", "NRMSE", "RMSE", "seconds"]].mean().round(3))

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
pcc_wide = fold_table.pivot(index="fold", columns="setting", values="PCC")
nrmse_wide = fold_table.pivot(index="fold", columns="setting", values="NRMSE")

paired_rows = []
for fold in range(N_BLOCKS):
    pcc_nopca = pcc_per_gene[(fold, "No PCA")]
    pcc_pca = pcc_per_gene[(fold, "PCA 256")]
    both_valid = ~np.isnan(pcc_nopca) & ~np.isnan(pcc_pca)
    genes_won = int(np.sum(pcc_nopca[both_valid] > pcc_pca[both_valid]))
    paired_rows.append({"fold": fold,
                        "PCC_diff": pcc_wide.loc[fold, "No PCA"] - pcc_wide.loc[fold, "PCA 256"],
                        "NRMSE_diff": nrmse_wide.loc[fold, "No PCA"] - nrmse_wide.loc[fold, "PCA 256"],
                        "genes_won_by_no_pca": genes_won,
                        "genes_compared": int(both_valid.sum())})
paired = pd.DataFrame(paired_rows)
print(paired.round(4).to_string(index=False))

mean_gain = paired["PCC_diff"].mean()
sd_gain = paired["PCC_diff"].std()
fold_wins = int(np.sum(paired["PCC_diff"] > 0))

print()
print(f"No PCA minus PCA 256, PCC: mean {mean_gain:+.4f}, sd {sd_gain:.4f}")
print(f"Folds won by no PCA: {fold_wins} / {N_BLOCKS}")
print()
if mean_gain >= MIN_PCC_GAIN and fold_wins >= MIN_FOLD_WINS:
    print("DECISION: no PCA passes the rule -> dropping PCA is justified on this slide.")
else:
    print("DECISION: no PCA does not pass the rule -> keep PCA 256.")

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
fold_table.to_csv(OUT_DIR / f"{sample_id}_blocks_per_fold.csv", index=False)
paired.to_csv(OUT_DIR / f"{sample_id}_blocks_paired.csv", index=False)
print("Saved CSVs to", OUT_DIR)

**Workflow overview:** This block performs the next stage of the spatial-block setup, buffered evaluation, scoring, or result plotting. The source run has no stored numerical result.

In [ ]:
model_settings = [f"PCA {k}" for k in PCA_SIZES] + ["No PCA"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(coords_kept[:, 0], coords_kept[:, 1], c=blocks, cmap="tab10", s=3)
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set_title("Spatial blocks")
axes[0].set_xticks([])
axes[0].set_yticks([])

x_positions = np.arange(len(model_settings))
for fold in range(N_BLOCKS):
    axes[1].plot(x_positions, [pcc_wide.loc[fold, s] for s in model_settings], marker="o", label=f"block {fold}")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(model_settings)
axes[1].set_ylabel("Mean PCC (test block)")
axes[1].set_title("PCC per held-out block")
axes[1].legend(fontsize=8)

axes[2].bar(paired["fold"], paired["PCC_diff"])
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].axhline(MIN_PCC_GAIN, color="red", linestyle="--", label=f"decision threshold (+{MIN_PCC_GAIN})")
axes[2].set_xlabel("Held-out block")
axes[2].set_ylabel("PCC: No PCA minus PCA 256")
axes[2].set_title("Paired difference (above 0 = no PCA better)")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / f"{sample_id}_blocks.png", dpi=150, bbox_inches="tight")
plt.show()

### How to read it

- **Compare with the random split.** There, no PCA beat PCA 256 by +0.007 PCC. If the mean difference here shrinks or turns negative, that gain was mostly no PCA memorizing neighbouring spots.
- **Look at the folds, not just the mean.** PCC varies a lot between blocks (different tissue in each region), which is why the comparison is paired: both settings are scored on the same block.
- **Genes won** shows whether a win is broad (most of the 50 genes) or driven by a few genes.
- **`alphas_at_grid_edge`** should be 0 or close to it. If many genes pick the smallest or largest λ, widen `ALPHAS`.
- **Absolute PCC will be lower** than in the random split. That is expected and is closer to the true performance.
- **Still one slide, one patient.** If no PCA passes the rule here, confirm with leave-one-patient-out across slides before switching.